In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, spearmanr, kendalltau
from sklearn import preprocessing
import scipy.stats as stats

In [2]:
data_violent_filt = pd.read_csv('data/cox-violent-parsed_filt.csv')
data_violent_filt

,id,name,first,last,sex,dob,age,age_cat,race,juv_fel_count,...,vr_charge_desc,type_of_assessment,decile_score.1,score_text,screening_date,v_type_of_assessment,v_decile_score,v_score_text,priors_count.1,event
0,1.0,miguel hernandez,miguel,hernandez,Male,18/04/1947,69,Greater than 45,Other,0,...,NaN,Risk of Recidivism,1,Low,14/08/2013,Risk of Violence,1,Low,0,0
1,2.0,miguel hernandez,miguel,hernandez,Male,18/04/1947,69,Greater than 45,Other,0,...,NaN,Risk of Recidivism,1,Low,14/08/2013,Risk of Violence,1,Low,0,0
2,3.0,michael ryan,michael,ryan,Male,06/02/1985,31,25 - 45,Caucasian,0,...,NaN,Risk of Recidivism,5,Medium,31/12/2014,Risk of Violence,2,Low,0,0
3,4.0,kevon dixon,kevon,dixon,Male,22/01/1982,34,25 - 45,African-American,0,...,Felony Battery (Dom Strang),Risk of Recidivism,3,Low,27/01/2013,Risk of Violence,1,Low,0,1
4,5.0,ed philo,ed,philo,Male,14/05/1991,24,Less than 25,African-American,0,...,NaN,Risk of Recidivism,4,Low,14/04/2013,Risk of Violence,3,Low,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18311,NaN,alexsandra beauchamps,alexsandra,beauchamps,Female,21/12/1984,31,25 - 45,African-American,0,...,NaN,Risk of Recidivism,6,Medium,29/12/2014,Risk of Violence,4,Low,5,0
18312,NaN,winston gregory,winston,gregory,Male,01/10/1958,57,Greater than 45,Other,0,...,NaN,Risk of Recidivism,1,Low,14/01/2014,Risk of Violence,1,Low,0,0
18313,NaN,farrah jean,farrah,jean,Female,17/11/1982,33,25 - 45,African-American,0,...,NaN,Risk of Recidivism,2,Low,09/03/2014,Risk of Violence,2,Low,3,0
18314,NaN,florencia sanmartin,florencia,sanmartin,Female,18/12/1992,23,Less than 25,Hispanic,0,...,NaN,Risk of Recidivism,4,Low,30/06/2014,Risk of Violence,4,Low,2,0


In [6]:
data_cleaned = data_violent_filt[['decile_score', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'c_jail_out', 'c_charge_degree', 'c_jail_in']]
data_cleaned['c_jail_in'] = pd.to_datetime(data_cleaned['c_jail_in'], format="%d/%m/%Y %H:%M")
data_cleaned['c_jail_out'] = pd.to_datetime(data_cleaned['c_jail_out'], format="%d/%m/%Y %H:%M")
data_cleaned['jail_time'] = (data_cleaned['c_jail_out'] - data_cleaned['c_jail_in']).dt.total_seconds() / (3600)

/tmp/ipykernel_18509/2073728006.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned['c_jail_in'] = pd.to_datetime(data_cleaned['c_jail_in'], format="%d/%m/%Y %H:%M")
/tmp/ipykernel_18509/2073728006.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned['c_jail_out'] = pd.to_datetime(data_cleaned['c_jail_out'], format="%d/%m/%Y %H:%M")
/tmp/ipykernel_18509/2073728006.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_ind

In [8]:
data_cleaned.drop(columns=['c_jail_in', 'c_jail_out'], inplace=True)

/tmp/ipykernel_18509/36494503.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned.drop(columns=['c_jail_in', 'c_jail_out'], inplace=True)


In [9]:
data_cleaned

,decile_score,juv_fel_count,juv_misd_count,juv_other_count,c_charge_degree,jail_time
0,1,0,0,0,(F3),23.633333
1,1,0,0,0,(F3),23.633333
2,5,0,0,0,NaN,NaN
3,3,0,0,0,(F3),241.850000
4,4,0,0,1,(F3),26.066667
...,...,...,...,...,...,...
18311,6,0,0,0,(M1),241.466667
18312,1,0,0,0,(F2),26.016667
18313,2,0,0,0,(M1),28.200000
18314,4,0,0,0,(F3),47.050000


In [13]:
list(data_cleaned['c_charge_degree'].unique())

['(F3)',
 nan,
 '(F7)',
 '(M1)',
 '(F2)',
 '(F1)',
 '(M2)',
 '(MO3)',
 '(X)',
 '(CT)',
 '(NI0)',
 '(F5)',
 '(TCX)',
 '(F6)',
 '(CO3)']

In [15]:
severity_rank = {
    "(X)": 1,
    "(NI0)": 2,
    "(CT)": 3,
    "(M03)": 4,
    "(M2)": 5,
    "(M1)": 6,
    "(CO3)": 7,
    "(TCX)": 8,
    "(F7)": 9,
    "(F6)": 10,
    "(F5)": 11,
    "(F3)": 12,
    "(F2)": 13,
    "(F1)": 14
}

In [19]:
data_cleaned = data_cleaned.applymap(lambda x: severity_rank.get(x) if x in severity_rank else x)

/tmp/ipykernel_18509/2103326274.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data_cleaned = data_cleaned.applymap(lambda x: severity_rank.get(x) if x in severity_rank else x)


In [20]:
data_cleaned.dropna(inplace=True)

In [21]:
data_cleaned

,decile_score,juv_fel_count,juv_misd_count,juv_other_count,c_charge_degree,jail_time
0,1,0,0,0,12,23.633333
1,1,0,0,0,12,23.633333
3,3,0,0,0,12,241.850000
4,4,0,0,1,12,26.066667
5,4,0,0,1,12,26.066667
...,...,...,...,...,...,...
18311,6,0,0,0,6,241.466667
18312,1,0,0,0,13,26.016667
18313,2,0,0,0,6,28.200000
18314,4,0,0,0,12,47.050000
